In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_squared_error, r2_score, mean_absolute_error

In [ ]:
#base de dados
#para previsão de tempo estimado de espera (base de dados: horário, dia da semana, tipo de consulta, tempo de espera)
bd_tmp_espera = {
    'horario': ['14:30', '20:00', '10:25'],
    'dia_semana': ['segunda-feira', 'quarta-feira', 'quinta-feira'],
    'tipo_consulta': ['Giconologista', 'Neurologista', 'Clinico-Geral'],
    'tempo_de_espera_min': [20, 10, 15]
}
#previsão de falta a consultas (no-show) (base de dados: nome, sexo, idade, consulta, histórico de atrasos (baixo, médio, elevado))
falta_consulta = {
    'nome': ['Jorge Henrique', 'Mariana Lima', 'Fernando Silva'],
    'sexo': ['Masculino', 'Feminino', 'Masculino'],
    'idade': [35, 24, 20],
    'consulta': ['Neurologista', 'Giconologista', 'Clinico-Geral'],
    'historico_atrasos': ['Médio', 'Baixo', 'elevado']
}

features_ideais = [
    'pacientes_na_fila_antes',      # Quantos pacientes esperando antes de você
    'pacientes_agendados_no_mesmo_horario',  # Conflitos de agenda
    'tempo_medio_consulta_tipo',     # Histórico: ortopedista = 25min em média
    'medico_habito_atraso',          # Histórico: Dr. X sempre atrasa 15min
    'horario_pico'                   # 8-10h e 14-16h geralmente mais cheio
]

bd_tmp_espera = {
    'horario': ['14:30', '20:00', '10:25'],
    'dia_semana': ['segunda-feira', 'quarta-feira', 'quinta-feira'],
    'tipo_consulta': ['Giconologista', 'Neurologista', 'Clinico-Geral'],
    'pacientes_na_fila_antes': [10, 5, 8],
    'pacientes_agendados_no_mesmo_horario': [2, 1, 3],
    'tempo_medio_consulta_tipo_min': [25, 15, 10],
    'medico_habito_atraso': [10, 5, 7],
    'horario_pico': ['sim', 'não', 'sim'],
    'tempo_de_espera_min': [20, 10, 15]
}

None


In [41]:

#Dados com mais sentido
np.random.seed(42)  # Para reprodutibilidade

# Função para gerar horários realistas
def gerar_horario():
    horas = np.random.choice([8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20], p=[0.12, 0.11, 0.10, 0.08, 0.12, 0.11, 0.10, 0.09, 0.07, 0.05, 0.03, 0.02])
    minutos = np.random.choice([0, 15, 30, 45])
    return f"{horas:02d}:{minutos:02d}"

# Dias da semana com pesos (segunda mais movimentada)
dias = ['segunda-feira', 'terça-feira', 'quarta-feira', 'quinta-feira', 'sexta-feira', 'sábado']
dias_pesos = [0.25, 0.20, 0.20, 0.18, 0.15, 0.02]

# Tipos de consulta com tempos médios reais
tipos_consulta = {
    'Clínico Geral': {'tempo_medio': 20, 'variacao': 10, 'peso': 0.30},
    'Pediatra': {'tempo_medio': 25, 'variacao': 12, 'peso': 0.15},
    'Ginecologista': {'tempo_medio': 30, 'variacao': 15, 'peso': 0.12},
    'Ortopedista': {'tempo_medio': 35, 'variacao': 18, 'peso': 0.10},
    'Dermatologista': {'tempo_medio': 20, 'variacao': 10, 'peso': 0.10},
    'Cardiologista': {'tempo_medio': 40, 'variacao': 20, 'peso': 0.08},
    'Neurologista': {'tempo_medio': 45, 'variacao': 22, 'peso': 0.06},
    'Oftalmologista': {'tempo_medio': 25, 'variacao': 12, 'peso': 0.04},
    'Psicólogo': {'tempo_medio': 50, 'variacao': 25, 'peso': 0.03},
    'Nutricionista': {'tempo_medio': 35, 'variacao': 15, 'peso': 0.02}
}

# Lista para armazenar os dados
dados = []

for i in range(100):
    # Seleciona tipo de consulta baseado nos pesos
    tipo = np.random.choice(list(tipos_consulta.keys()), p=[t['peso'] for t in tipos_consulta.values()])
    tempo_medio_tipo = tipos_consulta[tipo]['tempo_medio']
    variacao_tipo = tipos_consulta[tipo]['variacao']

    # Gera features
    horario = gerar_horario()
    hora = int(horario.split(':')[0])

    dia_semana = np.random.choice(dias, p=dias_pesos)

    # Pacientes na fila (varia por horário - mais de manhã e início da tarde)
    if hora < 10 or (14 <= hora < 16):
        pacientes_fila = int(np.random.normal(12, 4))
    elif hora < 12:
        pacientes_fila = int(np.random.normal(8, 3))
    else:
        pacientes_fila = int(np.random.normal(6, 2))
    pacientes_fila = max(1, min(25, pacientes_fila))

    # Pacientes agendados no mesmo horário (0-3 geralmente)
    pacientes_mesmo_horario = np.random.poisson(1.5)
    pacientes_mesmo_horario = min(4, pacientes_mesmo_horario)

    # Hábito de atraso do médico (varia por tipo e dia)
    atraso_base = np.random.normal(8, 5)
    if dia_semana == 'segunda-feira':
        atraso_base += 5
    if tipo in ['Cardiologista', 'Neurologista']:
        atraso_base += 10
    medico_habito_atraso = max(0, min(40, int(atraso_base)))

    # Horário de pico (manhã cedo ou começo da tarde)
    horario_pico = 'sim' if (hora < 10 or (14 <= hora < 16)) else 'não'

    # Cálculo do tempo de espera real (target)
    # Fórmula realista: fila * tempo_medio + atraso_médico + variação
    tempo_espera_base = (pacientes_fila * (tempo_medio_tipo / 60)) * 60  # converte para minutos
    tempo_espera_base += medico_habito_atraso

    # Adiciona variação baseada no tipo
    variacao = np.random.normal(0, variacao_tipo)

    # Efeito do dia da semana
    if dia_semana == 'segunda-feira':
        tempo_espera_base += 10
    elif dia_semana == 'sexta-feira':
        tempo_espera_base -= 5

    # Efeito do horário de pico
    if horario_pico == 'sim':
        tempo_espera_base += 8

    tempo_de_espera = max(5, int(tempo_espera_base + variacao))

    # Cria o registro
    registro = {
        'horario': horario,
        'dia_semana': dia_semana,
        'tipo_consulta': tipo,
        'pacientes_na_fila_antes': pacientes_fila,
        'pacientes_agendados_no_mesmo_horario': pacientes_mesmo_horario,
        'tempo_medio_consulta_tipo_min': tempo_medio_tipo,
        'medico_habito_atraso': medico_habito_atraso,
        'horario_pico': horario_pico,
        'tempo_de_espera_min': tempo_de_espera
    }

    dados.append(registro)

# CONVERSÃO PARA DATAFRAME
df = pd.DataFrame(dados)

# INSERINDO VALORES NULOS ESTRATÉGICOS (aproximadamente 10-15% nulos aleatórios)
colunas_com_nan = ['pacientes_na_fila_antes', 'pacientes_agendados_no_mesmo_horario',
                   'tempo_medio_consulta_tipo_min', 'medico_habito_atraso', 'horario_pico']

for col in colunas_com_nan:
    # Entre 5% e 15% de nulos por coluna
    nulos_idx = np.random.choice(df.index, size=int(len(df) * np.random.uniform(0.05, 0.15)), replace=False)
    df.loc[nulos_idx, col] = np.nan

# Adiciona alguns nulos no horario e dia_semana (raros)
nulos_horario = np.random.choice(df.index, size=3, replace=False)
df.loc[nulos_horario, 'horario'] = np.nan

nulos_dia = np.random.choice(df.index, size=2, replace=False)
df.loc[nulos_dia, 'dia_semana'] = np.nan

# ORDENA POR HORÁRIO (para fazer sentido temporal)
df = df.sort_values('horario').reset_index(drop=True)

# CONVERTE PARA O FORMATO DICIONÁRIO SOLICITADO
bd_tmp_espera = df.to_dict('list')
#df.to_csv('bd_tmp_espera2.csv', index=False)

In [ ]:

# EXIBINDO INFORMAÇÕES
print("="*60)
print("DATASET CRIADO COM SUCESSO!")
print("="*60)
print(f"\nTotal de registros: {len(df)}")
print(f"\nColunas: {list(df.columns)}")
print("\nValores nulos por coluna:")
print(df.isnull().sum())
print("\nPrimeiras 10 linhas:")
print(df.head(10))
print("\nEstatísticas do tempo de espera:")
print(df['tempo_de_espera_min'].describe())


In [ ]:
#Previsão de tempo estimado de espera
df = pd.read_csv('/content/bd_tmp_espera2.csv')
df.head(5)

In [31]:
x = df.drop('tempo_de_espera_min', axis=1)
y = df['tempo_de_espera_min'].values

In [32]:

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), ['horario', 'dia_semana', 'tipo_consulta', 'horario_pico']),
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ]), ['pacientes_na_fila_antes', 'pacientes_agendados_no_mesmo_horario',
             'tempo_medio_consulta_tipo_min', 'medico_habito_atraso'])
    ],
    remainder='drop'
)

In [33]:
X_transform = preprocessor.fit_transform(x)
rfr = RandomForestRegressor()

In [34]:
X_Train, X_Test, y_Train, y_Test = train_test_split(X_transform, y, test_size=0.3, random_state=42)

In [35]:
paramgrid = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [1.0, 'sqrt', 'log2', None],
    'bootstrap': [True, False]
}

In [ ]:

random_search = RandomizedSearchCV(
    estimator=rfr,
    param_distributions=paramgrid,
    n_iter=10,  # Number of parameter settings that are sampled
    cv=5,  # Number of cross-validation folds
    verbose=2,
    scoring="neg_mean_squared_error",
    random_state=42,
    n_jobs=-1  # Use all available processors
)
# Fit the RandomizedSearchCV instance
random_search.fit(X_Train, y_Train)

# Retrieve the best parameters found by RandomizedSearchCV
best_params = random_search.best_params_
best_params

In [ ]:
#Melhorar as features
random_forest_regressor = RandomForestRegressor(**best_params)
random_forest_regressor.fit(X_Train, y_Train)
test_predic = random_forest_regressor.predict(X_Test)
mse = mean_squared_error(y_Test, test_predic)
r2 = r2_score(y_Test, test_predic)
mae = mean_absolute_error(y_Test, test_predic)
print(mse)
print(r2)
print(mae)

In [ ]:
#plot de grafico com matplotlib
z = np.polyfit(y_Test, test_predic, 1)
p = np.poly1d(z)
plt.figure(figsize=(10, 6))
plt.scatter(y_Test, test_predic, alpha=0.6)
plt.plot(y_Test, p(y_Test), color='red', linestyle='--')
plt.xlabel('Valores Reais')
plt.ylabel('Valores Previstos')

In [ ]:
def gerar_horario():
    horas = np.random.choice([8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20], p=[0.12, 0.11, 0.10, 0.08, 0.12, 0.11, 0.10, 0.09, 0.07, 0.05, 0.03, 0.02])
    minutos = np.random.choice([0, 15, 30, 45])
    return f"{horas:02d}:{minutos:02d}"

# Dias da semana com pesos (segunda mais movimentada)
dias = ['segunda-feira', 'terça-feira', 'quarta-feira', 'quinta-feira', 'sexta-feira', 'sábado']
dias_pesos = [0.25, 0.20, 0.20, 0.18, 0.15, 0.02]

# Tipos de consulta com tempos médios reais
tipos_consulta = {
    'Clínico Geral': {'tempo_medio': 20, 'variacao': 10, 'peso': 0.30},
    'Pediatra': {'tempo_medio': 25, 'variacao': 12, 'peso': 0.15},
    'Ginecologista': {'tempo_medio': 30, 'variacao': 15, 'peso': 0.12},
    'Ortopedista': {'tempo_medio': 35, 'variacao': 18, 'peso': 0.10},
    'Dermatologista': {'tempo_medio': 20, 'variacao': 10, 'peso': 0.10},
    'Cardiologista': {'tempo_medio': 40, 'variacao': 20, 'peso': 0.08},
    'Neurologista': {'tempo_medio': 45, 'variacao': 22, 'peso': 0.06},
    'Oftalmologista': {'tempo_medio': 25, 'variacao': 12, 'peso': 0.04},
    'Psicólogo': {'tempo_medio': 50, 'variacao': 25, 'peso': 0.03},
    'Nutricionista': {'tempo_medio': 35, 'variacao': 15, 'peso': 0.02}
}

# Lista para armazenar os dados
dados = []

for i in range(20):
    # Seleciona tipo de consulta baseado nos pesos
    tipo = np.random.choice(list(tipos_consulta.keys()), p=[t['peso'] for t in tipos_consulta.values()])
    tempo_medio_tipo = tipos_consulta[tipo]['tempo_medio']
    variacao_tipo = tipos_consulta[tipo]['variacao']

    # Gera features
    horario = gerar_horario()
    hora = int(horario.split(':')[0])

    dia_semana = np.random.choice(dias, p=dias_pesos)

    # Pacientes na fila (varia por horário - mais de manhã e início da tarde)
    if hora < 10 or (14 <= hora < 16):
        pacientes_fila = int(np.random.normal(12, 4))
    elif hora < 12:
        pacientes_fila = int(np.random.normal(8, 3))
    else:
        pacientes_fila = int(np.random.normal(6, 2))
    pacientes_fila = max(1, min(25, pacientes_fila))

    # Pacientes agendados no mesmo horário (0-3 geralmente)
    pacientes_mesmo_horario = np.random.poisson(1.5)
    pacientes_mesmo_horario = min(4, pacientes_mesmo_horario)

    # Hábito de atraso do médico (varia por tipo e dia)
    atraso_base = np.random.normal(8, 5)
    if dia_semana == 'segunda-feira':
        atraso_base += 5
    if tipo in ['Cardiologista', 'Neurologista']:
        atraso_base += 10
    medico_habito_atraso = max(0, min(40, int(atraso_base)))

    # Horário de pico (manhã cedo ou começo da tarde)
    horario_pico = 'sim' if (hora < 10 or (14 <= hora < 16)) else 'não'

    # Cria o registro
    registro = {
        'horario': horario,
        'dia_semana': dia_semana,
        'tipo_consulta': tipo,
        'pacientes_na_fila_antes': pacientes_fila,
        'pacientes_agendados_no_mesmo_horario': pacientes_mesmo_horario,
        'tempo_medio_consulta_tipo_min': tempo_medio_tipo,
        'medico_habito_atraso': medico_habito_atraso,
        'horario_pico': horario_pico
    }

    dados.append(registro)


df_to_predict = pd.DataFrame(dados)
df_to_predict.to_csv('bd_tmp_espera_to_predict.csv', index=False)
df_to_predict.head(5)

In [69]:
df_predictions = pd.read_csv('/content/bd_tmp_espera_to_predict.csv')
df_to_predict_transform = preprocessor.transform(df_predictions)
predict = random_forest_regressor.predict(df_to_predict_transform)
df_to_predict['tempo_de_espera_min'] = predict
df_to_predict.head(5)
df_to_predict.to_csv('bd_tmp_espera_predict.csv', index=False)

In [ ]:

def gerar_horario():
    horas = np.random.choice([8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20], p=[0.12, 0.11, 0.10, 0.08, 0.12, 0.11, 0.10, 0.09, 0.07, 0.05, 0.03, 0.02])
    minutos = np.random.choice([0, 15, 30, 45])
    return f"{horas:02d}:{minutos:02d}"

# Dias da semana com pesos (segunda mais movimentada)
dias = ['segunda-feira', 'terça-feira', 'quarta-feira', 'quinta-feira', 'sexta-feira', 'sábado']
dias_pesos = [0.25, 0.20, 0.20, 0.18, 0.15, 0.02]

# Tipos de consulta com tempos médios reais
tipos_consulta = {
    'Clínico Geral': {'tempo_medio': 20, 'variacao': 10, 'peso': 0.30},
    'Pediatra': {'tempo_medio': 25, 'variacao': 12, 'peso': 0.15},
    'Ginecologista': {'tempo_medio': 30, 'variacao': 15, 'peso': 0.12},
    'Ortopedista': {'tempo_medio': 35, 'variacao': 18, 'peso': 0.10},
    'Dermatologista': {'tempo_medio': 20, 'variacao': 10, 'peso': 0.10},
    'Cardiologista': {'tempo_medio': 40, 'variacao': 20, 'peso': 0.08},
    'Neurologista': {'tempo_medio': 45, 'variacao': 22, 'peso': 0.06},
    'Oftalmologista': {'tempo_medio': 25, 'variacao': 12, 'peso': 0.04},
    'Psicólogo': {'tempo_medio': 50, 'variacao': 25, 'peso': 0.03},
    'Nutricionista': {'tempo_medio': 35, 'variacao': 15, 'peso': 0.02}
}

# Lista para armazenar os dados
dados = []

for i in range(20):
    # Seleciona tipo de consulta baseado nos pesos
    tipo = np.random.choice(list(tipos_consulta.keys()), p=[t['peso'] for t in tipos_consulta.values()])
    tempo_medio_tipo = tipos_consulta[tipo]['tempo_medio']
    variacao_tipo = tipos_consulta[tipo]['variacao']

    # Gera features
    horario = gerar_horario()
    hora = int(horario.split(':')[0])

    dia_semana = np.random.choice(dias, p=dias_pesos)

    # Hábito de atraso do médico (varia por tipo e dia)
    atraso_base = np.random.normal(8, 5)
    if dia_semana == 'segunda-feira':
        atraso_base += 5
    if tipo in ['Cardiologista', 'Neurologista']:
        atraso_base += 10
    medico_habito_atraso = max(0, min(40, int(atraso_base)))

    # Horário de pico (manhã cedo ou começo da tarde)
    horario_pico = 'sim' if (hora < 10 or (14 <= hora < 16)) else 'não'

    # Cria o registro | Onde está 0 é aonde precisa adicionar o conteúdo ainda
    registro = {
        'idade': 0,
        'consulta': tipo,
        'distacia_local_de_partida_e_consultorio': 0,
        'tipo_de_transporte': 0,
        'dia_semana': dia_semana,
        'horario_da_consulta': horario,
        'horario_pico': horario_pico,
        'PcD': 0,
        'histórico_de_atrasos': 0
    }

    dados.append(registro)

In [ ]:
#Criar o algoritmo de prever Construir um modelo preditivo para classificação de falta a
#consultas (no-show)
#(base de dados: idade, consulta, distacia_local_de_partida_e_consultorio, tipo_de_transporte (carro, oninus, moto, bicleta, caminhada),
#dia_semana, horario_da_consulta, horario_pico,PcD (sim, não),
#histórico_de_atrasos (baixo, médio, elevado))